# Leica TS16 local XML → Georeferenced export (GCP fit + QA)

This notebook does four things:

1. **Parses a Leica/LandXML XML** (or a previously exported dense CSV)
2. **Analyzes the GCP points** (duplicates, missing measured fields, possible label mistakes)
3. **Fits a local → global transformation** using Emlid GCP coordinates (default: Lambert 2008 + ellipsoidal)
4. **Exports the final table** with only:

- `point_oid`
- `timestamp`
- `Easting`
- `Northing`
- `Elevation`

It also reorders the table so that:

- **GCPs** (`1..8`) come first
- then **CP points** (`CP1`, `CP2`, …)
- then all other points

---

### Notes for your Merendree case
- You can remap mislabelled GCPs (e.g. local point `8` should match Emlid GCP `6`)
- The notebook handles missing `horiz_distance_m / slope_distance_m / angles` for some GCP rows
- Instrument / target heights are not required for the transformation (we fit directly from point coordinates)



In [1]:
# === USER CONFIG ===
from pathlib import Path

# 1) Leica input: XML (recommended) OR a previously exported dense CSV
LEICA_INPUT = r"L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MEENDRE2.xml"
# Example (already trimmed CSV):
# LEICA_INPUT = r"/mnt/data/200226_MEENDRE2_points_dense_trimmed.csv"

# 2) Emlid GCP CSV (must contain columns like Name, Easting, Northing, Elevation)
GCP_CSV = r"/mnt/data/200226_MerendreeATR.csv"

# 3) Where to save outputs (None = same folder as Leica input)
OUTPUT_DIR = None

# 4) GCP definitions (labels used in Leica point_oid)
GCP_IDS_EXPECTED = [str(i) for i in range(1, 9)]

# 5) Manual remap for known mistakes:
#    "local Leica point_oid" -> "Emlid GCP Name"
#    Your case: local point "8" is actually GCP "6"
GCP_ID_REMAP = {"8": "6"}

# Optional: if you want the final exported point_oid values also corrected (usually False)
APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT = False

# 6) Robust-fit settings (for bad/mislabeled GCPs)
H_RESIDUAL_INLIER_THRESH_M = 0.05   # 5 cm horizontal threshold for inliers
DUPLICATE_WARN_DIST_M = 0.05        # warn if two different local GCP labels are within 5 cm

# 7) Coordinate output settings
#    Emlid file in this case is already Lambert 2008 + ellipsoidal height.
EMLID_INPUT_HORIZONTAL_EPSG = 3812   # Lambert 2008
EMLID_INPUT_HEIGHT_MODE = "ellipsoidal"   # "ellipsoidal" or "mTAW"

# Final output target
OUTPUT_HORIZONTAL_EPSG = 3812        # 3812 (Lambert 2008) or 31370 (Lambert 72)
OUTPUT_HEIGHT_MODE = "ellipsoidal"   # "ellipsoidal" or "mTAW"

# 8) Belgian grid files (optional)
# If None, the notebook will try to find these in OUTPUT_DIR / input folder / current folder:
# - hBG18.geo
# - _BD72LB72_ETRS89LB08.gsb
GRID_DIR = None

# 9) Exports
EXPORT_CSV = True
EXPORT_XLSX = True
EXPORT_DIAGNOSTICS = True


In [2]:
# === Imports ===
import re
import itertools
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

# Optional (for CRS conversion after local->global fit)
try:
    import pyproj
    from pyproj import Transformer, datadir
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False
    pyproj = None
    Transformer = None
    datadir = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


In [3]:
# === Utility helpers ===

def normalize_point_id(v):
    """Normalize point IDs to readable strings (e.g. 1.0 -> '1')."""
    if pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s

def local_tag(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def to_float(v):
    try:
        if v in (None, "", " "):
            return None
        return float(v)
    except Exception:
        return None

def parse_xyz_text(text):
    if text is None:
        return (None, None, None)
    parts = str(text).strip().split()
    vals = [to_float(p) for p in parts[:3]]
    while len(vals) < 3:
        vals.append(None)
    return tuple(vals[:3])

def cp_number(pid):
    if pid is None:
        return None
    m = re.fullmatch(r"CP\s*0*([0-9]+)", str(pid), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def is_number_string(s):
    return bool(re.fullmatch(r"-?\d+(\.\d+)?", str(s).strip())) if s is not None else False

def fit_similarity(src, dst):
    """
    Umeyama similarity transform (N-D): dst ~= scale * R * src + t
    src, dst: (N, D)
    """
    src = np.asarray(src, dtype=float)
    dst = np.asarray(dst, dtype=float)
    n, d = src.shape
    mu_s = src.mean(axis=0)
    mu_d = dst.mean(axis=0)
    X = src - mu_s
    Y = dst - mu_d
    H = (X.T @ Y) / n
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    var_s = (X**2).sum() / n
    scale = float(S.sum() / var_s)
    t = mu_d - scale * (R @ mu_s)
    return scale, R, t

def apply_similarity(src, scale, R, t):
    src = np.asarray(src, dtype=float)
    return (scale * (src @ R.T)) + t

def rmse(v):
    v = np.asarray(v, dtype=float)
    return float(np.sqrt(np.mean(v**2))) if len(v) else np.nan


In [4]:
# === Parse Leica XML (or load existing dense CSV) ===

def parse_leica_xml_dense(xml_path: str) -> pd.DataFrame:
    """
    Parses Leica/LandXML and returns a dense table with local coordinates + (when present) raw obs fields.
    Keeps point_name internally for matching raw obs, but final export can omit it.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # RawObservation records grouped by TargetPoint name (preserve order for repeated names)
    raw_by_target_name = defaultdict(list)

    for ro in root.iter():
        if local_tag(ro.tag) != "RawObservation":
            continue

        target_point = None
        for ch in ro:
            if local_tag(ch.tag) == "TargetPoint":
                target_point = ch
                break

        target_name = target_point.attrib.get("name") if target_point is not None else None
        tx, ty, tz = parse_xyz_text(target_point.text if target_point is not None else None)

        raw_by_target_name[target_name].append({
            "obs_timestamp": ro.attrib.get("timeStamp"),
            "setup_id": ro.attrib.get("setupID"),
            "horiz_angle_gon": to_float(ro.attrib.get("horizAngle")),
            "zenith_angle_gon": to_float(ro.attrib.get("zenithAngle")),
            "horiz_distance_m": to_float(ro.attrib.get("horizDistance")),
            "slope_distance_m": to_float(ro.attrib.get("slopeDistance")),
            "direct_face": ro.attrib.get("directFace"),
            "target_height_m": to_float(ro.attrib.get("targetHeight")),
            "raw_target_name": target_name,
            "raw_x": tx, "raw_y": ty, "raw_z": tz,
        })

    rows = []
    for cp in root.iter():
        if local_tag(cp.tag) != "CgPoint":
            continue

        point_name = cp.attrib.get("name")
        point_oid = cp.attrib.get("oID") or cp.attrib.get("oid") or cp.attrib.get("id")
        timestamp = cp.attrib.get("timeStamp")
        purpose = cp.attrib.get("purpose")
        x, y, z = parse_xyz_text(cp.text)

        raw_match = None
        if point_name in raw_by_target_name and raw_by_target_name[point_name]:
            raw_match = raw_by_target_name[point_name].pop(0)

        row = {
            "point_oid": point_oid,
            "point_name": point_name,
            "timestamp": timestamp,
            "x": x, "y": y, "z": z,
            "purpose": purpose,
            "obs_timestamp": None,
            "setup_id": None,
            "horiz_angle_gon": None,
            "zenith_angle_gon": None,
            "horiz_distance_m": None,
            "slope_distance_m": None,
            "direct_face": None,
            "target_height_m": None,
            "raw_target_name": None,
            "raw_x": None, "raw_y": None, "raw_z": None,
        }
        if raw_match:
            row.update(raw_match)

        rows.append(row)

    df = pd.DataFrame(rows)
    if "timestamp" in df.columns:
        df = df.sort_values("timestamp", na_position="last").reset_index(drop=True)
    return df

def load_leica_dense(leica_input: str) -> pd.DataFrame:
    p = Path(leica_input)
    if not p.exists():
        raise FileNotFoundError(f"Leica input not found: {p}")

    if p.suffix.lower() == ".xml":
        print(f"Parsing XML: {p}")
        return parse_leica_xml_dense(str(p))
    elif p.suffix.lower() == ".csv":
        print(f"Loading dense CSV: {p}")
        df = pd.read_csv(p)
        # Ensure required columns exist
        required = {"point_oid", "timestamp", "x", "y", "z"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"CSV is missing required columns: {sorted(missing)}")
        # Add optional columns if absent (to simplify downstream code)
        for col in ["horiz_distance_m", "slope_distance_m", "horiz_angle_gon", "zenith_angle_gon", "setup_id"]:
            if col not in df.columns:
                df[col] = np.nan
        return df
    else:
        raise ValueError("LEICA_INPUT must be .xml or .csv")


In [5]:
# === Load Emlid GCP CSV ===

def load_emlid_gcps(gcp_csv_path: str) -> pd.DataFrame:
    p = Path(gcp_csv_path)
    if not p.exists():
        raise FileNotFoundError(f"GCP CSV not found: {p}")

    df = pd.read_csv(p)

    # flexible column detection
    colmap = {c.lower().strip(): c for c in df.columns}
    def find_col(candidates):
        for cand in candidates:
            for low, orig in colmap.items():
                if low == cand:
                    return orig
        for cand in candidates:
            for low, orig in colmap.items():
                if cand in low:
                    return orig
        return None

    c_name = find_col(["name", "point", "point_id", "id"])
    c_e = find_col(["easting", "x"])
    c_n = find_col(["northing", "y"])
    c_z = find_col(["elevation", "height", "z"])

    if not all([c_name, c_e, c_n, c_z]):
        raise ValueError(f"Could not detect GCP columns in {p.name}. Found columns: {list(df.columns)}")

    out = df[[c_name, c_e, c_n, c_z]].copy()
    out.columns = ["gcp_id_raw", "Easting", "Northing", "Elevation"]
    out["gcp_id"] = out["gcp_id_raw"].map(normalize_point_id)

    # numeric cleanup
    for c in ["Easting", "Northing", "Elevation"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=["gcp_id", "Easting", "Northing", "Elevation"]).reset_index(drop=True)
    return out



In [6]:
# === QA / GCP aggregation ===

def summarize_local_point_duplicates(local_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summary per point_oid over the full Leica dataset (counts, measured count, coordinate spread)
    """
    d = local_df.copy()
    d["point_oid_norm"] = d["point_oid"].map(normalize_point_id)

    rows = []
    for pid, sub in d.groupby("point_oid_norm", dropna=False):
        xyz = sub[["x", "y", "z"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(xyz):
            ctr = xyz.mean(axis=0).to_numpy()
            radial = np.sqrt(((xyz.to_numpy() - ctr)**2).sum(axis=1))
            spread_max = float(np.max(radial))
            sx, sy, sz = xyz.std(ddof=0).tolist()
            x_mean, y_mean, z_mean = xyz.mean().tolist()
        else:
            spread_max = np.nan
            sx = sy = sz = np.nan
            x_mean = y_mean = z_mean = np.nan

        rows.append({
            "point_oid": pid,
            "n_rows": len(sub),
            "n_measured_rows": int(pd.to_numeric(sub.get("slope_distance_m"), errors="coerce").notna().sum()),
            "first_timestamp": sub["timestamp"].min() if "timestamp" in sub.columns else None,
            "last_timestamp": sub["timestamp"].max() if "timestamp" in sub.columns else None,
            "x_mean": x_mean, "y_mean": y_mean, "z_mean": z_mean,
            "spread_max_m": spread_max,
            "std_x_m": sx, "std_y_m": sy, "std_z_m": sz,
        })

    out = pd.DataFrame(rows).sort_values(["n_rows", "point_oid"], ascending=[False, True], na_position="last").reset_index(drop=True)
    return out

def aggregate_gcp_local_points(local_df: pd.DataFrame, gcp_ids, prefer_measured=True) -> pd.DataFrame:
    """
    Build one representative local coordinate per GCP label from Leica point_oid.
    If measured rows exist for a label, use measured rows by default; otherwise use all rows.
    """
    d = local_df.copy()
    d["point_oid_norm"] = d["point_oid"].map(normalize_point_id)

    rows = []
    for pid in gcp_ids:
        sub_all = d[d["point_oid_norm"] == pid].copy()
        if sub_all.empty:
            rows.append({
                "local_id": pid, "exists_in_leica": False
            })
            continue

        meas_mask = pd.to_numeric(sub_all.get("slope_distance_m"), errors="coerce").notna()
        sub_used = sub_all[meas_mask].copy() if (prefer_measured and meas_mask.any()) else sub_all.copy()

        xyz_all = sub_all[["x", "y", "z"]].apply(pd.to_numeric, errors="coerce").dropna()
        xyz_used = sub_used[["x", "y", "z"]].apply(pd.to_numeric, errors="coerce").dropna()

        ctr_used = xyz_used.mean(axis=0)
        ctr_all = xyz_all.mean(axis=0)

        # spread of all rows for this point label (helps detect operator mistakes / repeated overwrites)
        if len(xyz_all):
            radial_all = np.sqrt(((xyz_all.to_numpy() - ctr_all.to_numpy())**2).sum(axis=1))
            spread_all = float(np.max(radial_all))
        else:
            spread_all = np.nan

        rows.append({
            "local_id": pid,
            "exists_in_leica": True,
            "n_rows_total": len(sub_all),
            "n_rows_measured": int(meas_mask.sum()),
            "n_rows_used": len(sub_used),
            "used_measured_rows": bool(meas_mask.any() and prefer_measured),
            "x_local": float(ctr_used["x"]),
            "y_local": float(ctr_used["y"]),
            "z_local": float(ctr_used["z"]),
            "spread_all_rows_m": spread_all,
            "timestamp_min": sub_all["timestamp"].min() if "timestamp" in sub_all.columns else None,
            "timestamp_max": sub_all["timestamp"].max() if "timestamp" in sub_all.columns else None,
            "has_horiz_distance": bool(pd.to_numeric(sub_all.get("horiz_distance_m"), errors="coerce").notna().any()),
            "has_slope_distance": bool(pd.to_numeric(sub_all.get("slope_distance_m"), errors="coerce").notna().any()),
            "has_horiz_angle": bool(pd.to_numeric(sub_all.get("horiz_angle_gon"), errors="coerce").notna().any()),
            "has_zenith_angle": bool(pd.to_numeric(sub_all.get("zenith_angle_gon"), errors="coerce").notna().any()),
        })

    return pd.DataFrame(rows)

def find_near_duplicate_gcps(gcp_local_df: pd.DataFrame, warn_thresh_m=0.05) -> pd.DataFrame:
    """
    Warn if two *different labels* are nearly identical in the local Leica system.
    This often reveals a mislabeled point (e.g. point 3 measured on point 2).
    """
    d = gcp_local_df.copy()
    d = d[d["exists_in_leica"] == True].copy()
    d = d.dropna(subset=["x_local", "y_local", "z_local"])

    rows = []
    for i in range(len(d)):
        for j in range(i + 1, len(d)):
            a = d.iloc[i]
            b = d.iloc[j]
            dist_3d = float(np.linalg.norm([
                a["x_local"] - b["x_local"],
                a["y_local"] - b["y_local"],
                a["z_local"] - b["z_local"]
            ]))
            dist_2d = float(np.linalg.norm([
                a["x_local"] - b["x_local"],
                a["y_local"] - b["y_local"]
            ]))
            if dist_3d <= warn_thresh_m:
                rows.append({
                    "local_id_a": a["local_id"],
                    "local_id_b": b["local_id"],
                    "local_dist_2d_m": dist_2d,
                    "local_dist_3d_m": dist_3d,
                    "warning": "Suspiciously close local coordinates for different GCP labels"
                })
    return pd.DataFrame(rows).sort_values(["local_dist_3d_m", "local_dist_2d_m"], ascending=[True, True]) if rows else pd.DataFrame(columns=["local_id_a","local_id_b","local_dist_2d_m","local_dist_3d_m","warning"])


In [7]:
# === Robust local -> global fit (horizontal + vertical) ===

def robust_fit_local_to_global(gcp_pairs_df: pd.DataFrame, h_inlier_thresh_m=0.05):
    """
    Finds a robust horizontal 2D similarity transform and vertical linear transform.
    Tries both local axis interpretations:
      - [x_local, y_local] -> [E, N]
      - [y_local, x_local] -> [E, N]
    and searches over subsets to reject mislabeled/outlier GCPs.

    Returns a dict with:
      - horizontal model
      - vertical model
      - inlier flags
      - residual tables
    """
    d = gcp_pairs_df.copy()
    d = d.dropna(subset=["x_local", "y_local", "z_local", "Easting", "Northing", "Elevation"]).reset_index(drop=True)
    if len(d) < 3:
        raise ValueError("Need at least 3 matched GCPs for a robust fit.")

    best = None
    idx_all = list(range(len(d)))

    for swap_xy in [False, True]:
        src_all = d[["y_local", "x_local"]].to_numpy() if swap_xy else d[["x_local", "y_local"]].to_numpy()
        dst_all = d[["Easting", "Northing"]].to_numpy()

        # subset search: 3..N
        for r in range(3, len(d) + 1):
            for subset in itertools.combinations(idx_all, r):
                src_sub = src_all[list(subset)]
                dst_sub = dst_all[list(subset)]

                # degenerate subset check
                if np.linalg.matrix_rank(src_sub - src_sub.mean(axis=0)) < 2:
                    continue

                try:
                    scale, R, t = fit_similarity(src_sub, dst_sub)
                except Exception:
                    continue

                pred_all = apply_similarity(src_all, scale, R, t)
                h_res = np.sqrt(((pred_all - dst_all)**2).sum(axis=1))
                inliers = h_res <= h_inlier_thresh_m

                n_in = int(inliers.sum())
                h_rmse_in = float(np.sqrt(np.mean(h_res[inliers]**2))) if n_in else np.inf
                h_rmse_all = float(np.sqrt(np.mean(h_res**2)))

                score = (n_in, -h_rmse_in, -h_rmse_all, r)  # maximize inliers, minimize RMSE

                if (best is None) or (score > best["score"]):
                    best = {
                        "score": score,
                        "swap_xy": swap_xy,
                        "h_scale": scale,
                        "h_R": R,
                        "h_t": t,
                        "h_residuals_m": h_res,
                        "inliers_h": inliers,
                        "subset_idx": subset,
                    }

    if best is None:
        raise RuntimeError("Could not fit a valid horizontal transform.")

    # Vertical model on horizontal inliers: z_global = a * z_local + b
    inlier_mask = best["inliers_h"]
    z_src = d.loc[inlier_mask, "z_local"].to_numpy(dtype=float)
    z_dst = d.loc[inlier_mask, "Elevation"].to_numpy(dtype=float)
    if len(z_src) < 2:
        # fallback: pure offset
        a, b = 1.0, float(np.nanmean(z_dst - z_src))
    else:
        A = np.vstack([z_src, np.ones_like(z_src)]).T
        a, b = np.linalg.lstsq(A, z_dst, rcond=None)[0]
        a = float(a)
        b = float(b)

    z_pred_all = d["z_local"].to_numpy(dtype=float) * a + b
    z_res = z_pred_all - d["Elevation"].to_numpy(dtype=float)

    # Build residual table
    out = d.copy()
    out["horizontal_residual_m"] = best["h_residuals_m"]
    out["vertical_residual_m"] = z_res
    out["inlier_h"] = inlier_mask
    out["pred_Easting"] = apply_similarity(
        d[["y_local", "x_local"]].to_numpy() if best["swap_xy"] else d[["x_local", "y_local"]].to_numpy(),
        best["h_scale"], best["h_R"], best["h_t"]
    )[:, 0]
    out["pred_Northing"] = apply_similarity(
        d[["y_local", "x_local"]].to_numpy() if best["swap_xy"] else d[["x_local", "y_local"]].to_numpy(),
        best["h_scale"], best["h_R"], best["h_t"]
    )[:, 1]
    out["pred_Elevation"] = z_pred_all

    result = {
        "pairs_table": out,
        "swap_xy": best["swap_xy"],
        "h_scale": best["h_scale"],
        "h_R": best["h_R"],
        "h_t": best["h_t"],
        "v_a": a,
        "v_b": b,
        "inlier_mask": inlier_mask,
    }
    return result

def leave_one_out_checkpoints(gcp_pairs_df: pd.DataFrame, inlier_mask: np.ndarray, swap_xy_fixed: bool, h_thresh_m=0.05):
    """
    Leave-one-out checkpoint errors on accepted inlier GCPs.
    This gives a rough estimate of generalization / transformation stability.
    """
    d = gcp_pairs_df.copy().reset_index(drop=True)
    d = d.loc[inlier_mask].reset_index(drop=True)
    if len(d) < 4:
        return pd.DataFrame(columns=["local_id", "target_id", "h_error_m", "z_error_m", "status"])

    rows = []
    for i in range(len(d)):
        train = d.drop(index=i).reset_index(drop=True)
        test = d.iloc[[i]].copy()

        src_train = train[["y_local", "x_local"]].to_numpy() if swap_xy_fixed else train[["x_local", "y_local"]].to_numpy()
        dst_train = train[["Easting", "Northing"]].to_numpy()
        src_test = test[["y_local", "x_local"]].to_numpy() if swap_xy_fixed else test[["x_local", "y_local"]].to_numpy()
        dst_test = test[["Easting", "Northing"]].to_numpy()

        # fit horizontal
        scale, R, t = fit_similarity(src_train, dst_train)
        pred_h = apply_similarity(src_test, scale, R, t)
        h_err = float(np.linalg.norm(pred_h[0] - dst_test[0]))

        # fit vertical
        z_src = train["z_local"].to_numpy(dtype=float)
        z_dst = train["Elevation"].to_numpy(dtype=float)
        A = np.vstack([z_src, np.ones_like(z_src)]).T
        a, b = np.linalg.lstsq(A, z_dst, rcond=None)[0]
        z_pred = float(test["z_local"].iloc[0] * a + b)
        z_err = float(z_pred - test["Elevation"].iloc[0])

        rows.append({
            "local_id": test["local_id"].iloc[0],
            "target_id": test["target_id"].iloc[0],
            "h_error_m": h_err,
            "z_error_m": z_err,
            "status": "checkpoint",
        })

    return pd.DataFrame(rows).sort_values("local_id", key=lambda s: s.astype(str))


In [8]:
# === Optional CRS / height conversion (Belgium) ===

def find_grid_files(grid_dir_hint, leica_input_path, output_dir):
    candidates = []
    if grid_dir_hint:
        candidates.append(Path(grid_dir_hint))
    if output_dir:
        candidates.append(Path(output_dir))
    candidates.append(Path(leica_input_path).parent)
    candidates.append(Path.cwd())

    seen = set()
    dirs = []
    for c in candidates:
        if c is None:
            continue
        c = Path(c)
        if c.exists():
            key = str(c.resolve())
            if key not in seen:
                dirs.append(c)
                seen.add(key)

    hbg18 = None
    ntv2 = None
    for d in dirs:
        p1 = d / "hBG18.geo"
        p2 = d / "_BD72LB72_ETRS89LB08.gsb"
        if p1.exists() and hbg18 is None:
            hbg18 = p1
        if p2.exists() and ntv2 is None:
            ntv2 = p2
    return hbg18, ntv2, dirs

def configure_pyproj_grid_search(dirs):
    if not HAVE_PYPROJ:
        return
    try:
        for d in dirs:
            datadir.append_data_dir(str(d))
    except Exception:
        pass

def convert_belgian_output_crs(df_in, src_epsg, dst_epsg, src_h_mode="ellipsoidal", dst_h_mode="ellipsoidal", hbg18_path=None):
    """
    Converts projected coordinates between Belgian CRSs and optionally ellipsoidal <-> mTAW heights.
    Horizontal conversion uses pyproj CRS transforms.
    Height conversion uses hBG18.geo if available.

    This is optional; in your Merendree case (Lambert 2008 + ellipsoidal -> same) it is a no-op.
    """
    if not HAVE_PYPROJ:
        if (src_epsg != dst_epsg) or (src_h_mode != dst_h_mode):
            raise RuntimeError("pyproj is required for CRS/height conversion but is not installed.")
        return df_in.copy()

    df = df_in.copy()

    # Horizontal conversion first (if needed)
    if int(src_epsg) != int(dst_epsg):
        t_h = Transformer.from_crs(int(src_epsg), int(dst_epsg), always_xy=True)
        e, n = t_h.transform(df["Easting"].to_numpy(), df["Northing"].to_numpy())
        df["Easting"] = e
        df["Northing"] = n

    # Height conversion (ellipsoidal <-> mTAW) using hBG18
    src_h_mode = str(src_h_mode).strip().lower()
    dst_h_mode = str(dst_h_mode).strip().lower()

    if src_h_mode != dst_h_mode:
        if hbg18_path is None or not Path(hbg18_path).exists():
            raise FileNotFoundError("Height conversion requested but hBG18.geo was not found.")

        # Convert current projected coords to ETRS89 geographic (EPSG:4258)
        # (For Lambert 72 this may internally use the available NTv2 grid if pyproj/proj can find it.)
        t_to_geo = Transformer.from_crs(int(dst_epsg), 4258, always_xy=True)
        lon, lat = t_to_geo.transform(df["Easting"].to_numpy(), df["Northing"].to_numpy())

        # Vertical grid shift in geographic coordinates
        # ellipsoidal -> mTAW  : apply vgridshift
        # mTAW -> ellipsoidal  : inverse vgridshift
        if src_h_mode == "ellipsoidal" and dst_h_mode == "mtaw":
            pipeline = f"+proj=pipeline +step +proj=vgridshift +grids={Path(hbg18_path).as_posix()}"
        elif src_h_mode == "mtaw" and dst_h_mode == "ellipsoidal":
            pipeline = f"+proj=pipeline +step +inv +proj=vgridshift +grids={Path(hbg18_path).as_posix()}"
        else:
            raise ValueError("Height mode must be 'ellipsoidal' or 'mTAW'.")

        t_v = Transformer.from_pipeline(pipeline)
        lon2, lat2, z2 = t_v.transform(lon, lat, df["Elevation"].to_numpy())

        # lon/lat should be unchanged; keep z result
        df["Elevation"] = z2

    return df


In [9]:
# === Main workflow ===

leica_path = Path(LEICA_INPUT)
if OUTPUT_DIR is None:
    out_dir = leica_path.parent
else:
    out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Load Leica and Emlid
leica_df = load_leica_dense(str(leica_path))
gcp_df = load_emlid_gcps(GCP_CSV)

# Normalize point IDs
leica_df["point_oid_norm"] = leica_df["point_oid"].map(normalize_point_id)
gcp_df["gcp_id"] = gcp_df["gcp_id"].map(normalize_point_id)

print(f"Leica rows: {len(leica_df):,}")
print(f"Emlid GCP rows: {len(gcp_df):,}")
print("Leica columns:", list(leica_df.columns))
print("Emlid GCP columns:", list(gcp_df.columns))


Parsing XML: L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MEENDRE2.xml


FileNotFoundError: GCP CSV not found: \mnt\data\200226_MerendreeATR.csv

In [ ]:
# === Diagnostics: duplicates and GCP availability ===

dup_summary = summarize_local_point_duplicates(leica_df)

# Show duplicates / repeated point_oids first
dup_repeats = dup_summary[dup_summary["n_rows"] > 1].copy()
print("Repeated point_oids (top 30):")
display(dup_repeats.head(30))

# Aggregate expected GCPs (1..8 by default)
gcp_local = aggregate_gcp_local_points(leica_df, GCP_IDS_EXPECTED, prefer_measured=True)

# Attach Emlid target IDs (with manual remap)
gcp_local["target_id"] = gcp_local["local_id"].replace(GCP_ID_REMAP)
gcp_pairs = gcp_local.merge(
    gcp_df[["gcp_id", "Easting", "Northing", "Elevation"]],
    left_on="target_id",
    right_on="gcp_id",
    how="left"
)

print("\nExpected GCPs in Leica + Emlid match status:")
display(gcp_pairs[[
    "local_id","target_id","exists_in_leica","n_rows_total","n_rows_measured","n_rows_used","used_measured_rows",
    "has_horiz_distance","has_slope_distance","has_horiz_angle","has_zenith_angle",
    "x_local","y_local","z_local","spread_all_rows_m","Easting","Northing","Elevation"
]])

# Near-duplicate GCP labels in local coordinates (useful for human error detection)
near_dup_gcps = find_near_duplicate_gcps(gcp_local, warn_thresh_m=DUPLICATE_WARN_DIST_M)
if len(near_dup_gcps):
    print("\n⚠️ Suspiciously close GCP labels in Leica local coordinates:")
    display(near_dup_gcps)
else:
    print("\nNo suspiciously close GCP label pairs found (within threshold).")


In [ ]:
# === Fit robust local -> global transformation ===

# Candidate pairs: only rows that exist in Leica and have an Emlid target match
cand = gcp_pairs.copy()
cand = cand[(cand["exists_in_leica"] == True)].copy()
cand = cand.dropna(subset=["Easting","Northing","Elevation"]).reset_index(drop=True)

fit = robust_fit_local_to_global(cand, h_inlier_thresh_m=H_RESIDUAL_INLIER_THRESH_M)

fit_table = fit["pairs_table"].copy()
fit_table["status"] = np.where(fit_table["inlier_h"], "inlier", "outlier")

print("Axis interpretation chosen by fit:")
print("  Leica local [y, x] -> [Easting, Northing]" if fit["swap_xy"] else "  Leica local [x, y] -> [Easting, Northing]")
print(f"Horizontal similarity scale: {fit['h_scale']:.9f}")
print(f"Vertical model: Elevation = {fit['v_a']:.9f} * z_local + {fit['v_b']:.6f}")

print("\nGCP residuals (local labels vs Emlid targets):")
display(fit_table[[
    "local_id","target_id","status",
    "x_local","y_local","z_local",
    "Easting","Northing","Elevation",
    "pred_Easting","pred_Northing","pred_Elevation",
    "horizontal_residual_m","vertical_residual_m"
]].sort_values(["status","local_id"]))

inlier_mask = fit["inlier_mask"]
n_in = int(np.sum(inlier_mask))
print(f"\nAccepted horizontal inliers: {n_in} / {len(cand)}")
print(f"Horizontal RMSE (inliers): {rmse(fit_table.loc[fit_table['inlier_h'],'horizontal_residual_m']):.4f} m")
print(f"Vertical RMSE (inliers):   {rmse(fit_table.loc[fit_table['inlier_h'],'vertical_residual_m']):.4f} m")

if (~inlier_mask).any():
    print("\nOutlier GCP labels rejected from the transform (likely mislabeled / wrong correspondence):")
    display(fit_table.loc[~fit_table["inlier_h"], ["local_id","target_id","horizontal_residual_m","vertical_residual_m"]])


In [ ]:
# === Leave-one-out checkpoint test on accepted GCPs ===

loo = leave_one_out_checkpoints(cand, fit["inlier_mask"], fit["swap_xy"], h_thresh_m=H_RESIDUAL_INLIER_THRESH_M)

if len(loo):
    print("Leave-one-out checkpoint errors (fit on all accepted GCPs except one, then predict the held-out point):")
    display(loo)
    print(f"LOO horizontal RMSE: {rmse(loo['h_error_m']):.4f} m")
    print(f"LOO vertical RMSE:   {rmse(loo['z_error_m']):.4f} m")
else:
    print("Not enough accepted GCPs for leave-one-out checkpoint testing.")


In [ ]:
# === Apply transformation to ALL Leica points ===

# Horizontal transform
src_all = leica_df[["y","x"]].to_numpy(dtype=float) if fit["swap_xy"] else leica_df[["x","y"]].to_numpy(dtype=float)
EN = apply_similarity(src_all, fit["h_scale"], fit["h_R"], fit["h_t"])

# Vertical transform
Zg = leica_df["z"].to_numpy(dtype=float) * fit["v_a"] + fit["v_b"]

geo_df = leica_df.copy()
geo_df["Easting"] = EN[:, 0]
geo_df["Northing"] = EN[:, 1]
geo_df["Elevation"] = Zg

# Optional output CRS / height conversion
hbg18_path, ntv2_path, grid_dirs_scanned = find_grid_files(GRID_DIR, leica_path, out_dir)
if HAVE_PYPROJ:
    configure_pyproj_grid_search(grid_dirs_scanned)

print("Grid search dirs:")
for d in grid_dirs_scanned:
    print(" -", d)
print("hBG18.geo:", hbg18_path)
print("_BD72LB72_ETRS89LB08.gsb:", ntv2_path)

geo_df = convert_belgian_output_crs(
    geo_df,
    src_epsg=EMLID_INPUT_HORIZONTAL_EPSG,
    dst_epsg=OUTPUT_HORIZONTAL_EPSG,
    src_h_mode=EMLID_INPUT_HEIGHT_MODE,
    dst_h_mode=OUTPUT_HEIGHT_MODE,
    hbg18_path=hbg18_path
)

# Optional point ID remap in final output (usually leave False to preserve raw Leica labels)
if APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT:
    geo_df["point_oid_export"] = geo_df["point_oid_norm"].replace(GCP_ID_REMAP)
else:
    geo_df["point_oid_export"] = geo_df["point_oid"].astype(str)

# Sorting: GCPs first, then CP*, then everything else
gcp_order = {str(pid): i for i, pid in enumerate(GCP_IDS_EXPECTED, start=1)}
geo_df["gcp_sort_idx"] = geo_df["point_oid_norm"].map(gcp_order)
geo_df["cp_sort_idx"] = geo_df["point_oid_norm"].map(cp_number)

geo_df["sort_group"] = np.where(
    geo_df["gcp_sort_idx"].notna(), 0,
    np.where(geo_df["cp_sort_idx"].notna(), 1, 2)
)

geo_df = geo_df.sort_values(
    ["sort_group", "gcp_sort_idx", "cp_sort_idx", "timestamp"],
    na_position="last"
).reset_index(drop=True)

# Final output columns only
final_df = geo_df[["point_oid_export", "timestamp", "Easting", "Northing", "Elevation"]].copy()
final_df = final_df.rename(columns={"point_oid_export": "point_oid"})

print(f"Final rows: {len(final_df):,}")
display(final_df.head(40))


In [ ]:
# === Export files ===

stem = Path(LEICA_INPUT).stem
suffix = f"_georef_EPSG{OUTPUT_HORIZONTAL_EPSG}_{OUTPUT_HEIGHT_MODE}"

final_csv = out_dir / f"{stem}{suffix}.csv"
final_xlsx = out_dir / f"{stem}{suffix}.xlsx"
diag_gcp_csv = out_dir / f"{stem}_gcp_fit_diagnostics.csv"
diag_summary_csv = out_dir / f"{stem}_pointoid_duplicate_summary.csv"
diag_report_txt = out_dir / f"{stem}_transform_report.txt"

if EXPORT_CSV:
    final_df.to_csv(final_csv, index=False)

if EXPORT_XLSX:
    # Only final output columns in the final xlsx (as requested)
    final_df.to_excel(final_xlsx, index=False)

if EXPORT_DIAGNOSTICS:
    fit_table.to_csv(diag_gcp_csv, index=False)
    dup_summary.to_csv(diag_summary_csv, index=False)

    # Human-readable report
    inlier_fit = fit_table[fit_table["inlier_h"]].copy()
    loo_h_rmse = rmse(loo["h_error_m"]) if len(loo) else np.nan
    loo_z_rmse = rmse(loo["z_error_m"]) if len(loo) else np.nan

    lines = []
    lines.append("Leica local -> georeferenced transformation report")
    lines.append("=" * 60)
    lines.append(f"Leica input: {Path(LEICA_INPUT)}")
    lines.append(f"GCP CSV:     {Path(GCP_CSV)}")
    lines.append("")
    lines.append("Manual GCP remap:")
    lines.append(str(GCP_ID_REMAP))
    lines.append("")
    lines.append("Axis interpretation:")
    lines.append("Local [y, x] -> [Easting, Northing]" if fit["swap_xy"] else "Local [x, y] -> [Easting, Northing]")
    lines.append(f"Horizontal scale: {fit['h_scale']:.9f}")
    lines.append(f"Vertical model: Elevation = {fit['v_a']:.9f} * z_local + {fit['v_b']:.6f}")
    lines.append("")
    lines.append(f"Accepted GCP inliers: {int(inlier_fit.shape[0])} / {int(fit_table.shape[0])}")
    lines.append(f"Horizontal RMSE (inliers): {rmse(inlier_fit['horizontal_residual_m']):.4f} m")
    lines.append(f"Vertical RMSE (inliers):   {rmse(inlier_fit['vertical_residual_m']):.4f} m")
    if len(loo):
        lines.append(f"LOO Horizontal RMSE:       {loo_h_rmse:.4f} m")
        lines.append(f"LOO Vertical RMSE:         {loo_z_rmse:.4f} m")
    lines.append("")
    outliers = fit_table[~fit_table["inlier_h"]]
    if len(outliers):
        lines.append("Rejected GCP labels (outliers):")
        for _, r in outliers.iterrows():
            lines.append(f" - local {r['local_id']} -> target {r['target_id']} | h={r['horizontal_residual_m']:.3f} m, z={r['vertical_residual_m']:.3f} m")
    else:
        lines.append("No GCP outliers rejected.")
    lines.append("")
    if len(near_dup_gcps):
        lines.append("Suspicious near-duplicate local GCP labels:")
        for _, r in near_dup_gcps.iterrows():
            lines.append(f" - {r['local_id_a']} vs {r['local_id_b']} | local 3D distance={r['local_dist_3d_m']:.4f} m")
    lines.append("")
    lines.append("Output:")
    lines.append(f" - {final_csv}" if EXPORT_CSV else " - CSV export skipped")
    lines.append(f" - {final_xlsx}" if EXPORT_XLSX else " - XLSX export skipped")
    lines.append(f" - {diag_gcp_csv}")
    lines.append(f" - {diag_summary_csv}")

    diag_report_txt.write_text("\n".join(lines), encoding="utf-8")

print("Done.")
print("Final CSV :", final_csv if EXPORT_CSV else "(skipped)")
print("Final XLSX:", final_xlsx if EXPORT_XLSX else "(skipped)")
if EXPORT_DIAGNOSTICS:
    print("GCP diagnostics:", diag_gcp_csv)
    print("PointOID summary:", diag_summary_csv)
    print("Report TXT:", diag_report_txt)


### Interpretation tips

- If the notebook reports a **near-duplicate pair** (e.g. local GCP `2` and `3` almost identical), that is usually a strong sign of a **mislabelled Leica point**.
- The robust fit will usually reject that point as an **outlier** automatically.
- In the Merendree file, it is expected that:
  - local Leica axes are likely interpreted as **`[y, x] -> [Easting, Northing]`**
  - local point **`8` should map to Emlid GCP `6`** (via `GCP_ID_REMAP`)
- The final export is intentionally minimal:
  - `point_oid, timestamp, Easting, Northing, Elevation`
